# *<center> Digital quadrupole: the stability diagrams </center>*

**Purpose.** This notebook maps the stability diagram of a quadrupole driven by a two-level (digital) square wave, where the **duty cycle** — the fraction of the period the x pair spends at the high level — becomes a second operating knob that the sinusoidal drive does not have. It reproduces the four operating zones of Hu and Gundlach-Graham, *Rapid Commun. Mass Spectrom.* 38 (2024) e9753 (Table 1), gates the drawn bands against that table, verifies one band in flight, prices the timing precision that band demands from the integrator — the same precision a hardware switch would need — and records what flying the 3-D deck of record shows. The companion notebook `sin_stability.ipynb` draws the sinusoidal baseline these diagrams are compared against.

## Background

With a square drive the Mathieu equation becomes the **Meissner equation**: in $\tau = \Omega t / 2$,

$$\ddot{u} + \left(a_u - 2 q_u\, w(\tau)\right) u = 0,$$

where $w$ is a two-level square of duty $d$ — high for a fraction $d$ of the period, low for $1-d$. The y pair carries the exact inverse of the x waveform, so the y axis sees duty $1-d$ with the high window starting where x goes low, and a filter transmits only where **both** axes are stable: x at $(a, q, d)$, y at $(-a, q, 1-d)$.

Two things make the digital diagram different from the sinusoidal one. First, the coefficient is piecewise constant, so the one-period transfer matrix — the 2×2 matrix that maps an ion's position and velocity across one drive period, whose trace alone decides stability because its determinant is pinned to one; `sin_stability.ipynb` derives this in its background section — is a **product of two analytic segment propagators** (cos/cosh blocks) — `meissner_half_trace` evaluates it exactly, with no integration grid and no error at the switch instants. Second, the duty cycle **moves and reshapes the stability bands**: at duty 0.5 the higher bands are the narrow curiosities the sinusoidal notebook found, and away from 0.5 they shift to where a mass filter can (in principle) operate — Table 1's zones sit at $q$ = 4.59 to 12.5, with band-limited resolving powers $R_{BL}$ = 556 to 12 500 set by the band widths themselves.

The widths are the whole story. A band of relative width $\Delta q / q \sim 2 \times 10^{-3}$ (zone 3,1) means every knob that touches $q$ — drive voltage, frequency, the duty cycle, the field's own curvature — must hold to a fraction of that. This notebook measures those demands rather than asserting them: the duty tolerance comes from the transfer matrix, the timing requirement from a flown integration-step ladder, and the field-calibration requirement from what the 3-D deck does.

Nothing drawn below is transcribed from a reference. Every curve and band is computed when the notebook runs — the transfer matrices by `ion_gym.physics.mathieu`, the flown points by the same solver and integrator any ion_gym deck uses, the voltage calibration from the solved field of the quad being flown. The published Table 1 enters exactly once, as the gate the computed numbers are checked against.

## The instruments

Two devices appear below. The **2-D analytic rod quad** of `quad_lib.py` (four round rods, $r_0$ = 3 mm, 0.1 mm lattice) flies every transmission scan: it is the same quad the sinusoidal notebook verified, so a flown band here is attributable to the drive, not to a new geometry. The **3-D STL deck of record**, `decks/quadrupole_digital_3d.json`, is the shipped digital variant of the repo's 3-D quadrupole; its rods sit on the diagonals and its drive frequency is calibrated against its own solved field (section 8). The deck is rendered below from the solver's voxelization, in the doctrine's three orthogonal views.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# ---- session resources: throughput choices, not physics -----------------
# Every flight in this notebook is a single ion, so a worker count would
# change nothing here and none is set. The trajectory-record budget below
# is passed to every build_run, as the workbench passes its own.
from ion_gym.physics.sizing import default_record_budget_gb, resources_line

RECORD_BUDGET_GB = default_record_budget_gb()   # GB the trajectory record
                                   # may use: RECORD_REFUSE_FRACTION of RAM,
                                   # the workbench default. build_run warns
                                   # at half of it and refuses above it.
                                   # None = RAM unreadable, fixed limits.
print(resources_line(None, RECORD_BUDGET_GB))

In [ ]:
# the shared machinery lives beside this notebook in the shipped
# manuscript_examples folder
import sys
from ion_gym.io.paths import examples_dir, repo_root
STUDY_DIR = repo_root() / "manuscript_examples" / "quadrupole"
if str(STUDY_DIR) not in sys.path:
    sys.path.insert(0, str(STUDY_DIR))

import numpy as np
import matplotlib.pyplot as plt
import time

import quad_lib as QL
from make_digital_quad_deck import ZONES
from ion_gym.physics.mathieu import hill_half_trace_njit, meissner_half_trace
from ion_gym.progress import quote, track

# one chart style and palette for both stability notebooks
QL.apply_chart_style()
CLR = QL.CHART_COLORS

In [ ]:
# ---- parameters ---------------------------------------------------------
MZ = 100.0              # Th; the flown mass in the 2-D quad
F_RF_MHZ = 1.0          # 2-D drive frequency
V_CAL = 30.0            # 2-D calibration amplitude

ZONE = "3,1"            # the zone every flown section operates in
DECK = STUDY_DIR / "decks" / "quadrupole_digital_3d.json"

# the duty-q overview map at a = 0
ND_MAP, NQ_MAP = 181, 601

# section 2, first figure: the third panel's duty (not 0.5, whose
# panel sits beside it)
DUTY_SHIFTED = 0.40

# zone 1,1 at a shifted duty (section 2, second figure), after Brabeck
# and Reilly, J. Am. Soc. Mass Spectrom. 27 (2016) 1122, Figure 3e/3f:
# their 61.124/38.876 waveform and the resolving power they report for it
DUTY_PAPER_LONG = 0.61124   # the longer of the two segments
R_PAPER = 100.0             # their q / dq on the a = 0 line
SHIFT_Q, SHIFT_A = (0.0, 1.0), (-0.50, 0.10)     # full-zone window
N_SHIFT = 181
ZOOM_Q, ZOOM_A = (0.580, 0.600), (-0.010, 0.010)  # apex window
N_ZOOM = 201
DUTY_SPAN = (0.50, 0.95)
Q_SPAN = (0.0, 16.0)

# per-zone (a, q) windows, scaled to EACH zone's own a = 0 band width
# (a fixed relative window left zone 3,3's band sub-pixel and its panel
# blank): q spans +/- Q_WIN_WIDTHS band widths, a spans +/- A_WIN_WIDTHS
ZONE_REL_SPAN = 0.02    # band-search window around the Table-1 q
Q_WIN_WIDTHS = 25.0
A_WIN_WIDTHS = 50.0
N_ZONE = 121            # grid per zone panel (both axes)

# flown band scan (2-D quad)
N_PHASE = 16            # entry phases per transmission estimate
REL_SCAN = np.linspace(0.9960, 1.0040, 17)   # q / q_center
T_MAX_US = 50.0         # ~26 drive periods at 1 MHz
DT_NS = 0.1             # resolves the duty edge to 1e-4 of the period

# integration-step ladders at band center. Aligned: dt = T/N with N a
# multiple of 400 = lcm(200, 16) so every switch edge AND every entry
# phase of the 16-phase ladder lands exactly on a step. Jitter: values
# chosen NOT to divide the 1000 ns period or the 835 ns edge.
DT_ALIGNED_N = [14400, 4000, 2000, 1600, 800, 400]
DT_JITTER_NS = [0.07, 0.13, 0.29, 0.61, 1.3]

# 3-D deck lifetime scan
REL_F_3D = np.arange(0.9990, 1.00301, 0.00025)  # f / f_deck
N_PHASE_3D = 8
T_MAX_3D_US = 30.0

In [ ]:
# the 3-D deck of record, drawn from the solver's own voxelization in
# three orthogonal views (xy / xz / yz), and its inherited settings
# reported rather than left buried in the JSON
from ion_gym.io.sim_spec import SimSpec
from ion_gym.io.deck_params import describe_deck
from ion_gym.physics.sim_build import build_run
from ion_gym.viz.viz_core import scene_from_model3d, render_mpl

spec3d = SimSpec.from_json(str(DECK))
describe_deck(spec3d)
model3d, fly3d, cols3d, _births3d = build_run(spec3d, record_budget_gb=RECORD_BUDGET_GB)
scene = scene_from_model3d(spec3d, model3d,
                           title=f"{spec3d.name} — solved at "
                                 f"{spec3d.geometry.mm_per_gu} mm/gu")
# render_mpl deregisters its figure from pyplot (no double display);
# the returned figure's rich repr IS the display, so it ends the cell
render_mpl(scene)

## 1 · Calibration of the 2-D quad

Identical to the sinusoidal notebook: one curvature read from the solved field converts $q$ to volts, and $q$ scales as $1/\Omega^2$ between frequencies. Every flown $q$ below is a voltage derived from this number.

In [ ]:
# one field solve prices the whole chart in volts
t0 = time.time()
V_PER_Q, q_cal = QL.calibrate_v_per_q(f_rf_mhz=F_RF_MHZ, v_cal=V_CAL, mz=MZ)
print(f"V_PER_Q = {V_PER_Q:.2f} V per unit q at {F_RF_MHZ} MHz "
      f"(q = {q_cal:.4f} at {V_CAL} V; {time.time()-t0:.0f}s)")

## 2 · Zone 1,1 under the two drives

Before the duty cycle becomes a knob, hold it at 0.5 and put the square drive next to the sinusoid on the sinusoid's home ground: the first stability zone, $x$ band 1 crossing $y$ band 1 — the tongue every quadrupole text draws. The first two maps below share axes, color scale and margin convention, and both are computed point by point from their transfer matrices, so every difference between the panels is physics of the waveform and nothing else. The square drive's tongue has the same shape with its boundaries compressed in $q$, because a square wave of amplitude $V$ carries a fundamental of amplitude $4V/\pi$: at zone 1 the higher harmonics matter little, so the square-driven quad behaves like a sinusoidal one run $4/\pi$ harder. The printed edge and tip values come from the maps themselves, and their ratio reproduces $4/\pi$ to a tenth of a percent — a number a reader can check against the maps by eye. The color convention is the sinusoidal notebook's, read the same way: red is the orbit's per-period amplitude gain, blue inside the zone is $|\cos \pi\beta|$ (deepest where $\beta = 1/2$, which is not a well depth), and each black line is the $|\mathrm{tr}\,M|/2 = 1$ level of its own computed map. White is not a third state: the scale passes through white exactly at $|\mathrm{tr}\,M|/2 = 1$, so the white fringe **outside** the black line is weak instability — amplitude gain barely above one per period. An ion there is lost eventually but can outlive any finite transit, which is why a real filter of finite length transmits partially just outside the boundary and why resolution improves with residence time; the black line is sharp only at infinite time. The $a = 0$ edge of each zone is marked with its value, located by bisection on the transfer matrix rather than read off the pixel grid.

The third map moves the square drive's duty to 0.4 and keeps everything else the same. The waveform now spends more of each period at $-V$ than at $+V$, so its average is no longer zero, and the region shifts down in $a$. The $a = 0$ line then crosses the region only over a short stretch of $q$, and both ends of that stretch are marked. The subsection after the figure explains the shift and follows it to the duty at which the apex reaches $a = 0$.

In [ ]:
# the same (a, q) window under each drive; sin uses the RK4 transfer
# matrix, the square drive the exact two-segment product
N_11 = 161
A_11 = 0.50       # +/-; the duty-shifted region reaches a = -0.45
Q_11 = 1.25
if DUTY_SHIFTED == 0.5:
    raise ValueError("DUTY_SHIFTED must differ from 0.5: the duty-0.5 "
                     "panel is already drawn beside it")

# warm both kernels, then price from the slower (RK4) one
hill_half_trace_njit(0.1, 0.5)
meissner_half_trace(0.1, 0.5, 0.5)
probe_t0 = time.time()
hill_half_trace_njit(0.1, 0.5)
hill_half_trace_njit(-0.1, 0.5)
per_point_s = time.time() - probe_t0
quote("zone 1,1 comparison maps (priced at the RK4 per-point time; the "
      "exact square-drive maps cost far less)", 3 * N_11 * N_11,
      per_item_s=per_point_s)

aa11 = np.linspace(-A_11, A_11, N_11)
qq11 = np.linspace(0.0, Q_11, N_11)
m_sin = np.zeros((N_11, N_11))
m_dig = np.zeros((N_11, N_11))
m_dig2 = np.zeros((N_11, N_11))
for i, a in track(list(enumerate(aa11)), label="1,1 rows"):
    for j, q in enumerate(qq11):
        m_sin[i, j] = np.log10(max(abs(hill_half_trace_njit(a, q)),
                                   abs(hill_half_trace_njit(-a, q)),
                                   1e-12))
        m_dig[i, j] = np.log10(max(abs(meissner_half_trace(a, q, 0.5)),
                                   abs(meissner_half_trace(-a, q, 0.5)),
                                   1e-12))
        m_dig2[i, j] = np.log10(max(
            abs(meissner_half_trace(a, q, DUTY_SHIFTED)),
            abs(meissner_half_trace(-a, q, 1.0 - DUTY_SHIFTED)), 1e-12))

# the numbers the panels claim, read from the physics, not from a
# text: the a = 0 instability edge of each zone, bisected on the
# transfer matrix to 1e-6 (the pixel grid only locates it to ~0.008)
def a0_edge(trace_fn, q_in, q_out):
    # q_in stable, q_out unstable, on either side of each other (an
    # upper edge has q_out above q_in, a lower edge below); bisect
    # |tr M|/2 = 1 to 1e-6
    if not abs(trace_fn(q_in)) < 1.0:
        raise ValueError(f"bisection start q = {q_in} is not stable")
    if not abs(trace_fn(q_out)) > 1.0:
        raise ValueError(f"bisection start q = {q_out} is not unstable")
    while abs(q_out - q_in) > 1e-6:
        mid = 0.5 * (q_in + q_out)
        if abs(trace_fn(mid)) < 1.0:
            q_in = mid
        else:
            q_out = mid
    return 0.5 * (q_in + q_out)

edge_sin = a0_edge(lambda q: hill_half_trace_njit(0.0, q), 0.85, 0.95)
edge_dig = a0_edge(lambda q: meissner_half_trace(0.0, q, 0.5), 0.65, 0.78)
Q1_TEXTBOOK = 0.908          # the standard Mathieu zone-1 edge value
print(f"sinusoidal a = 0 edge: q = {edge_sin:.4f} computed vs "
      f"{Q1_TEXTBOOK} textbook")
print(f"square (duty 0.5) a = 0 edge: q = {edge_dig:.4f} computed vs "
      f"{edge_sin * np.pi / 4:.4f} from scaling the sinusoidal edge "
      f"by the fundamental's pi/4")
print(f"edge ratio sin/square = {edge_sin / edge_dig:.4f} vs "
      f"4/pi = {4 / np.pi:.4f} "
      f"({abs(edge_sin / edge_dig / (4 / np.pi) - 1):.2%} apart)")

# the duty-shifted region crosses a = 0 over one stretch of q: find it
# on a coarse scan, then bisect both of its ends on the two-axis trace
def two_axis_trace(q):
    return max(abs(meissner_half_trace(0.0, q, DUTY_SHIFTED)),
               abs(meissner_half_trace(0.0, q, 1.0 - DUTY_SHIFTED)))

runs = QL.a0_stable_bands(lambda q: two_axis_trace(q) < 1.0, 1e-4, 1.2,
                          n=1201)
if not runs:
    raise ValueError(f"duty {DUTY_SHIFTED}: no two-axis stable q on a = 0 "
                     f"(the band has closed, see the next subsection)")
run_lo, run_hi = runs[0]
scan_step = (1.2 - 1e-4) / 1200
run_mid = 0.5 * (run_lo + run_hi)
shift_lo = a0_edge(two_axis_trace, run_mid, run_lo - scan_step)
shift_hi = a0_edge(two_axis_trace, run_mid, run_hi + scan_step)
print(f"square (duty {DUTY_SHIFTED}) a = 0 band: q = {shift_lo:.4f} to "
      f"{shift_hi:.4f}")

for name, m in (("sinusoidal", m_sin), ("square, duty 0.5", m_dig),
                (f"square, duty {DUTY_SHIFTED}", m_dig2)):
    stable = m < 0.0
    ij = np.argwhere(stable)
    a_st = aa11[ij[:, 0]]
    hi_t, lo_t = ij[np.argmax(a_st)], ij[np.argmin(a_st)]
    print(f"{name}: tips near (q = {qq11[hi_t[1]]:.3f}, "
          f"a = {aa11[hi_t[0]]:+.3f}) and (q = {qq11[lo_t[1]]:.3f}, "
          f"a = {aa11[lo_t[0]]:+.3f})")

from matplotlib.colors import TwoSlopeNorm
vmin = min(m_sin.min(), m_dig.min(), m_dig2.min())
vmax = np.percentile(np.concatenate([m_sin, m_dig, m_dig2]), 60)
norm = TwoSlopeNorm(vcenter=0.0, vmin=vmin, vmax=vmax)
fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.4), sharey=True,
                         layout="constrained")
# label heights in a: the shifted panel's apex sits just above a = 0,
# so its label goes higher to clear the boundary
for ax, m, title, edges, label_a in (
        (axes[0], m_sin, "sinusoidal drive", (edge_sin,), 0.035),
        (axes[1], m_dig, "two-level square, duty 0.5", (edge_dig,), 0.035),
        (axes[2], m_dig2, f"two-level square, duty {DUTY_SHIFTED}",
         (shift_lo, shift_hi), 0.085)):
    pm = ax.pcolormesh(qq11, aa11, m, cmap="RdBu_r", norm=norm,
                       shading="auto", rasterized=True)
    ax.contour(qq11, aa11, m, levels=[0.0], colors="k", linewidths=1.2)
    ax.axhline(0, color="k", lw=0.5, alpha=0.6)
    for edge in edges:
        ax.plot(edge, 0.0, "o", color="black", ms=5,
                markerfacecolor="white", markeredgewidth=1.4, zorder=5)
    label = (f"q = {edges[0]:.4f}" if len(edges) == 1
             else f"q = {edges[0]:.4f} to {edges[1]:.4f}")
    QL.halo_text(ax, edges[-1] + 0.025, label_a, label,
                 fontsize=10, fontweight="semibold", color="black")
    ax.grid(False)
    ax.set_title(f"zone 1,1 — {title}")
    ax.set_xlabel("q")
axes[0].set_ylabel("a")
fig.colorbar(pm, ax=axes, label="log10 max(|tr M|/2) over x, y",
             shrink=0.9)
fig.suptitle("the first stability zone under the sinusoidal drive and "
             "the square drive at two duties, same margin convention "
             "(computed at run time)", fontsize=11)
plt.show()

### A 61.124/38.876 duty cycle places the zone 1,1 apex on the a = 0 line

The square drive above spends equal time at $+V$ and $-V$, so its average over one period is zero and the zone 1,1 region is symmetric about $a = 0$. Moving the duty $d$ off 0.5 gives the waveform a nonzero average of $(2d - 1)V$. That average acts on each axis like a DC term. It adds to $a$ on one axis and subtracts from it on the other, because the $y$ rods see the inverted waveform at duty $1 - d$. The region therefore shifts in $a$. In this notebook's convention a duty above 0.5 raises the region and a duty below 0.5 lowers it, and swapping $d$ for $1 - d$ reflects the map in $a$.

Brabeck and Reilly (J. Am. Soc. Mass Spectrom. 27 (2016) 1122) used this shift as a second way to run a mass filter. Their 61.124/38.876 waveform lowers the region until its upper apex sits just above $a = 0$. The $a = 0$ line then crosses the region only near the apex, so only a narrow range of $q$, and therefore of $m/z$, is stable, with no DC on the rods. They report a resolving power $R = q/\Delta q = 100$ at that duty. The figure below reproduces their Figures 3e and 3f in this notebook's map format. The left panel is the full region at $d = 0.38876$, the duty in this notebook's convention that lowers the region as their waveform does. The right panel enlarges the apex, and the stable stretch of the $a = 0$ line is marked at its two edges, each located by bisection on the transfer matrix. The printed checks compare the computed resolving power and apex position with the paper and give the duty at which the apex reaches $a = 0$ exactly, where the stable stretch closes.

In [ ]:
# zone 1,1 of the square drive at the paper's shifted duty: the a = 0
# band edges by bisection, the duty where they meet, the full map and
# the apex enlargement
from scipy.optimize import brentq
from matplotlib.colors import TwoSlopeNorm

def first_stable_q(d):
    # a q inside the first stable band on a = 0 of the axis at duty d,
    # the midpoint of the first stable run of a coarse scan; bisection
    # then locates that band's lower edge exactly
    runs = QL.a0_stable_bands(
        lambda q: abs(meissner_half_trace(0.0, q, d)) < 1.0, 1e-4, 1.2,
        n=1201)
    if not runs:
        raise ValueError(f"no stable band on a = 0 at duty {d}")
    return 0.5 * (runs[0][0] + runs[0][1])

def band_a0(d_long):
    # (q_lo, q_hi) of the two-axis zone 1,1 band on a = 0 for a waveform
    # whose longer segment is d_long > 0.5: the axis at duty d_long sets
    # the lower edge, the axis at 1 - d_long the upper. The band on a = 0
    # is the same for d and 1 - d (the map reflects in a). q_lo > q_hi
    # means the band has closed.
    q_lo = a0_edge(lambda q: meissner_half_trace(0.0, q, d_long),
                   first_stable_q(d_long), 1e-4)
    q_hi = a0_edge(lambda q: meissner_half_trace(0.0, q, 1.0 - d_long),
                   0.05, 0.95)
    return q_lo, q_hi

q_lo, q_hi = band_a0(DUTY_PAPER_LONG)
q_c = 0.5 * (q_lo + q_hi)
r_calc = q_c / (q_hi - q_lo)
paper_label = (f"{100 * DUTY_PAPER_LONG:.3f}/"
               f"{100 * (1 - DUTY_PAPER_LONG):.3f}")
print(f"waveform {paper_label}: a = 0 band "
      f"q = {q_lo:.5f} to {q_hi:.5f}, R = q/dq = {r_calc:.1f} computed "
      f"vs {R_PAPER:g} reported ({abs(r_calc / R_PAPER - 1):.1%} apart)")
# the band is open at 0.60 and closed at 0.62 (scanned while the section
# was written), which brackets the duty where its edges meet
d_crit = brentq(lambda d: band_a0(d)[1] - band_a0(d)[0], 0.60, 0.62,
                xtol=1e-7)
print(f"the apex reaches a = 0 exactly at duty {d_crit:.5f}/"
      f"{1 - d_crit:.5f}, where the band closes at "
      f"q = {np.mean(band_a0(d_crit)):.4f} and R grows without bound")

d_map = 1.0 - DUTY_PAPER_LONG          # this notebook's convention
meissner_half_trace(0.1, 0.5, d_map)
probe_t0 = time.time()
for _ in range(200):
    meissner_half_trace(0.1, 0.5, d_map)
    meissner_half_trace(-0.1, 0.5, 1.0 - d_map)
quote("full and apex maps (exact two-segment transfer matrix)",
      N_SHIFT * N_SHIFT + N_ZOOM * N_ZOOM,
      per_item_s=(time.time() - probe_t0) / 200)

def margin_map(qq, aa, d):
    # log10 of the larger |tr M|/2 over the x axis (a, q, d) and the
    # y axis (-a, q, 1 - d), the convention of the figure above
    m = np.zeros((aa.size, qq.size))
    for i, a in enumerate(aa):
        for j, q in enumerate(qq):
            m[i, j] = np.log10(max(abs(meissner_half_trace(a, q, d)),
                                   abs(meissner_half_trace(-a, q,
                                                           1.0 - d)),
                                   1e-12))
    return m

qq_s = np.linspace(*SHIFT_Q, N_SHIFT)
aa_s = np.linspace(*SHIFT_A, N_SHIFT)
qq_z = np.linspace(*ZOOM_Q, N_ZOOM)
aa_z = np.linspace(*ZOOM_A, N_ZOOM)
m_shift = margin_map(qq_s, aa_s, d_map)
m_zoom = margin_map(qq_z, aa_z, d_map)
ij = np.argwhere(m_zoom < 0.0)
tip = ij[np.argmax(aa_z[ij[:, 0]])]
print(f"apex near (q = {qq_z[tip[1]]:.4f}, a = {aa_z[tip[0]]:+.4f}) "
      f"(grid step {aa_z[1] - aa_z[0]:.1e} in a)")

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.4), layout="constrained")
panels = ((axes[0], qq_s, aa_s, m_shift,
           f"zone 1,1 — two-level square, duty {d_map:.5f}"),
          (axes[1], qq_z, aa_z, m_zoom, "the apex, enlarged"))
for ax, qq, aa, m, title in panels:
    # each panel its own diverging scale centered on the boundary: the
    # apex window lies within a few percent of |tr M|/2 = 1, so the
    # full panel's scale would draw it white throughout
    norm = TwoSlopeNorm(vcenter=0.0, vmin=m.min(),
                        vmax=max(np.percentile(m, 60), 1e-6))
    pm = ax.pcolormesh(qq, aa, m, cmap="RdBu_r", norm=norm,
                       shading="auto", rasterized=True)
    ax.contour(qq, aa, m, levels=[0.0], colors="k", linewidths=1.2)
    ax.axhline(0, color="k", lw=0.6, alpha=0.7)
    ax.grid(False)
    ax.set_title(title)
    ax.set_xlabel("q")
    ax.set_ylabel("a")
    fig.colorbar(pm, ax=ax, label="log10 max(|tr M|/2) over x, y",
                 shrink=0.9)
axes[0].add_patch(plt.Rectangle((ZOOM_Q[0], ZOOM_A[0]),
                                ZOOM_Q[1] - ZOOM_Q[0],
                                ZOOM_A[1] - ZOOM_A[0], fill=False,
                                ec=CLR["zone_mark"], lw=1.2))
for q_edge in (q_lo, q_hi):
    axes[1].plot(q_edge, 0.0, "o", color="black", ms=5,
                 markerfacecolor="white", markeredgewidth=1.4, zorder=5)
QL.halo_text(axes[1], q_c, -0.0055,
             f"q = {q_lo:.4f} to {q_hi:.4f}\nR = q/dq = {r_calc:.1f}",
             fontsize=10, fontweight="semibold", color="black",
             ha="center", va="top")
fig.suptitle(f"zone 1,1 lowered by the {paper_label} waveform: the apex "
             f"sits on a = 0 (computed at run time)", fontsize=11)
plt.show()

## 3 · The a = 0 band structure against duty cycle

The map below answers the question the sinusoidal notebook ends on: what does the duty knob do to the high-$q$ bands? Plotting stable/unstable directly would erase the answer — the Table-1 bands are $10^{-3}$ of $q$ wide, far below any drawable pixel — so the color is the **stability margin**: $\log_{10}$ of the larger of the two axes' $|\mathrm{tr}\,M / 2|$, which is $< 0$ (blue) exactly where both axes are stable and, in the red, is the orbit's amplitude gain per drive period — a growth rate, not a well depth (the pseudopotential never appears on this map). Read the pale ridges carefully: they are **not** operating curves. Along a ridge one axis is close to stable, but the filter needs both, and the scan in the next section shows there is no two-axis stable $q$ anywhere at, say, duty 0.620 or 0.630 — one step off zone 3,2's duty. Both-axes stability exists only on small **islands** in $(d, q)$, and the black segments mark those islands, each drawn at its exact computed $a=0$ band (stretched to the panel's minimum visible width, since the real widths are sub-pixel). The next section shows why the islands sit where they do.

In [ ]:
# the margin map: log10 of the worse axis's |half trace| at a = 0.
# Exactly < 0 where x AND y are both stable; the exact per-zone bands
# are overlaid so sub-pixel valleys still carry their operating points.
# warm the njit kernel first: a cold probe times the COMPILE, and the
# quoted estimate would be two orders of magnitude high
QL.digital_stable(0.0, 8.0, 0.7)
probe_t0 = time.time()
QL.digital_stable(0.0, 8.0, 0.7)
per_point_s = time.time() - probe_t0
quote("exact Meissner margin map (measured per-point)", ND_MAP * NQ_MAP,
      per_item_s=per_point_s)

duties = np.linspace(*DUTY_SPAN, ND_MAP)
qq = np.linspace(*Q_SPAN, NQ_MAP)
margin = np.zeros((ND_MAP, NQ_MAP))
for i, d in track(list(enumerate(duties)), label="duty rows"):
    for j, q in enumerate(qq):
        hx = abs(meissner_half_trace(0.0, q, d))
        hy = abs(meissner_half_trace(0.0, q, 1.0 - d))
        margin[i, j] = np.log10(max(hx, hy, 1e-12))

from matplotlib.colors import TwoSlopeNorm
fig, ax = plt.subplots(figsize=(9.4, 5.2), layout="constrained")
# cap the red scale below the deep-unstable maximum so the near-stable
# ridges carry the contrast
vmax = np.percentile(margin, 85)
pm = ax.pcolormesh(qq, duties, margin, cmap="RdBu_r",
                   norm=TwoSlopeNorm(vcenter=0.0, vmin=margin.min(),
                                     vmax=vmax), shading="auto",
                   rasterized=True)
fig.colorbar(pm, ax=ax, label="log10 max(|tr M|/2) over x, y "
                              "(< 0 = both stable)")
for name, z in ZONES.items():
    lo_z, hi_z = QL.zone_band_a0(z["duty"], z["q"])
    half = max((hi_z - lo_z) / 2, 0.012 * (Q_SPAN[1] - Q_SPAN[0]) / 2)
    # black on a white underlay, so the marks read on any map color
    ax.plot([z["q"] - half, z["q"] + half], [z["duty"]] * 2, "-",
            color="white", lw=5.0, solid_capstyle="butt")
    ax.plot([z["q"] - half, z["q"] + half], [z["duty"]] * 2, "-",
            color="black", lw=2.6, solid_capstyle="butt")
    QL.halo_text(ax, z["q"] + 1.6 * half, z["duty"], name,
                 fontsize=9.5, fontweight="semibold",
                 color="black", va="center")
ax.grid(False)
ax.set_xlabel("q")
ax.set_ylabel("duty cycle d (x pair high fraction)")
ax.set_title(f"digital drive, a = 0: stability margin "
             f"({ND_MAP}x{NQ_MAP} exact transfer matrices)\n"
             f"black segments: the four Table-1 operating islands, "
             f"each at its exact a = 0 band (min. visible width)",
             fontsize=10)
plt.show()

## 4 · Why the zones sit where they do: crossings, not a ridge

Each transverse axis is its own Meissner system with its own family of stable bands: the $x$ axis at duty $d$, the $y$ axis at duty $1-d$. As $d$ rises, the $x$ bands move one way in $q$ and the $y$ bands the **other** way, because their duties change in opposite directions. The filter needs both axes at once, so an operating zone exists only where an $x$ band **crosses** a $y$ band — and the paper's zone label "$m{,}n$" is exactly that pair: $x$ band $m$ crossing $y$ band $n$. The crossings are small islands in $(d, q)$, not curves. The cell below measures each island by following its crossing until one axis genuinely fails: zone 3,1's island spans about 0.015 in duty, 4,1's 0.008, 3,2's 0.001, and 3,3's collapses toward a point — at duty 0.5 the two axes see the same waveform, so 3,3 is a band crossing **itself**, tangentially. Within an island the operating $q$ is not fixed either: it slides along the $x$ band as the duty moves (17% across zone 3,1's island), so duty and $q$ must be set **together**. One display matter is worth naming so a reader does not misread it: the overlap looks sparse or dotted on any raster, because the island's $q$ width is $10^{-3}$ to $10^{-4}$ of $q$ — thinner than a pixel — and a sampled image catches it only where a sample row lands inside. The island itself is a **continuous** strip, so the panels draw it as a solid curve taken from the crossing-following computation (the same one that prints the extents), not from the raster. This is the duty-precision statement of section 7 seen geometrically, and it answers the question the overview map raises: the pale ridges are single-axis near-stability, and no operating point exists along them away from the islands.

In [ ]:
# measure each island by following its crossing: step the duty, track
# the x band's center, and keep the part of the x band that is also
# y-stable -- the island's own cross-section at that duty. The window
# travels with the band, so the extent is the island's, not a window's.
N_X_SCAN = 20001
N_Y_CHECK = 201
ISLAND_DUTY_STEP = 2.5e-4

def x_band_near(duty, q_hint, span):
    qs = np.linspace(q_hint - span, q_hint + span, N_X_SCAN)
    ok = np.array([abs(meissner_half_trace(0.0, q, duty)) < 1.0
                   for q in qs])
    idx = np.nonzero(ok)[0]
    if len(idx) == 0:
        return None
    brk = np.nonzero(np.diff(idx) > 1)[0]
    starts = [idx[0]] + [idx[b + 1] for b in brk]
    ends = [idx[b] for b in brk] + [idx[-1]]
    runs = [(qs[a], qs[b]) for a, b in zip(starts, ends)]
    return min(runs, key=lambda r: abs(0.5 * (r[0] + r[1]) - q_hint))

def island_cross_section(duty, band):
    # the y-stable part of the x band at this duty, or None
    qs_in = np.linspace(band[0], band[1], N_Y_CHECK)
    oky = np.array([abs(meissner_half_trace(0.0, q, 1.0 - duty)) < 1.0
                    for q in qs_in])
    idx = np.nonzero(oky)[0]
    if len(idx) == 0:
        return None
    return float(qs_in[idx[0]]), float(qs_in[idx[-1]])

def follow_island(zone):
    z = ZONES[zone]
    span = 0.06 * z["q"]
    sections = {}
    for direction in (+1, -1):
        d = z["duty"]
        q_hint = z["q"]
        while True:
            band = x_band_near(d, q_hint, span)
            if band is None:
                break
            sect = island_cross_section(d, band)
            if sect is None:
                break
            sections[round(d, 6)] = sect
            q_hint = 0.5 * (band[0] + band[1])
            d = d + direction * ISLAND_DUTY_STEP
    dd = np.array(sorted(sections))
    lo = np.array([sections[d][0] for d in dd])
    hi = np.array([sections[d][1] for d in dd])
    return dd, lo, hi

probe_t0 = time.time()
x_band_near(ZONES["3,1"]["duty"], ZONES["3,1"]["q"], 0.06 * 4.59)
per_step_s = time.time() - probe_t0
quote("island-following steps across the four zones (first step "
      "measured; step count is the measured extents / step size)",
      120, per_item_s=per_step_s)
islands = {}
print(f"{'zone':>5} {'duty_lo':>9} {'duty_hi':>9} {'extent':>8} "
      f"{'q at lo':>8} {'q at hi':>8}")
for name in ZONES:
    dd, lo, hi = follow_island(name)
    islands[name] = (dd, lo, hi)
    print(f"{name:>5} {dd[0]:9.4f} {dd[-1]:9.4f} {dd[-1] - dd[0]:8.4f} "
          f"{0.5 * (lo[0] + hi[0]):8.3f} {0.5 * (lo[-1] + hi[-1]):8.3f}")

In [ ]:
# one panel per zone: the x-stable and y-stable stripes in (duty, q),
# with the island drawn as the CONTINUOUS followed curve from the cell
# above -- its q width (1e-3 to 1e-4 of q) is below one pixel, so a
# raster alone would show it as disconnected dots
from matplotlib.colors import ListedColormap
N_D_ISLAND, N_Q_ISLAND = 81, 801
WINDOW_PAD = 1.3            # panel half-extent / measured half-extent
D_HALF_FLOOR = 0.0015
Q_HALF_FLOOR_REL = 0.02

probe_t0 = time.time()
meissner_half_trace(0.0, 4.59, 0.835)
meissner_half_trace(0.0, 4.59, 0.165)
per_pair_s = time.time() - probe_t0
quote("island meshes, both axes per point (measured per-point)",
      4 * N_D_ISLAND * N_Q_ISLAND, per_item_s=per_pair_s)

cmx = ListedColormap([CLR["x_band"]])
cmy = ListedColormap([CLR["y_band"]])
fig, axes = plt.subplots(2, 2, figsize=(10.4, 7.6), layout="constrained")
for ax, (name, z) in zip(axes.ravel(), ZONES.items()):
    dd, lo, hi = islands[name]
    ctr_q = 0.5 * (lo[0] + hi[0] + lo[-1] + hi[-1]) / 2
    d_ctr = 0.5 * (dd[0] + dd[-1])
    d_half = max(WINDOW_PAD * 0.5 * (dd[-1] - dd[0]), D_HALF_FLOOR)
    q_half = max(WINDOW_PAD * 0.5 * abs(0.5 * (lo[-1] + hi[-1])
                                        - 0.5 * (lo[0] + hi[0])),
                 Q_HALF_FLOOR_REL * ctr_q)
    dmesh = np.linspace(d_ctr - d_half, d_ctr + d_half, N_D_ISLAND)
    qmesh = np.linspace(ctr_q - q_half, ctr_q + q_half, N_Q_ISLAND)
    stx = np.zeros((N_D_ISLAND, N_Q_ISLAND), dtype=bool)
    sty = np.zeros_like(stx)
    for i, d in enumerate(dmesh):
        for j, q in enumerate(qmesh):
            stx[i, j] = abs(meissner_half_trace(0.0, q, d)) < 1.0
            sty[i, j] = abs(meissner_half_trace(0.0, q, 1.0 - d)) < 1.0
    for st_ax, cm in ((stx, cmx), (sty, cmy)):
        masked = np.ma.masked_where(~st_ax, st_ax.astype(float))
        ax.pcolormesh(qmesh, dmesh, masked, cmap=cm, alpha=0.38,
                      shading="auto", rasterized=True)
    # the island itself: the followed strip, drawn continuous
    if len(dd) > 1:
        ax.fill_betweenx(dd, lo, hi, color=CLR["island"], lw=0)
        ax.plot(0.5 * (lo + hi), dd, "-", color=CLR["island"], lw=2.2,
                solid_capstyle="round")
    else:
        ax.plot(0.5 * (lo + hi), dd, "D", color=CLR["island"], ms=7)
    ax.plot(z["q"], z["duty"], "o", color=CLR["zone_mark"], ms=7,
            markeredgecolor="white", markeredgewidth=1.2, zorder=5)
    ax.grid(False)
    ax.set_title(f"zone {name}: island exists only for d in "
                 f"[{dd[0]:.4f}, {dd[-1]:.4f}]", fontsize=10)
    ax.set_xlabel("q")
    ax.set_ylabel("duty cycle d")
handles = [plt.Rectangle((0, 0), 1, 1, color=CLR["x_band"], alpha=0.38),
           plt.Rectangle((0, 0), 1, 1, color=CLR["y_band"], alpha=0.38),
           plt.Line2D([], [], color=CLR["island"], lw=2.2),
           plt.Line2D([], [], marker="o", ls="", color=CLR["zone_mark"],
                      markeredgecolor="white")]
fig.legend(handles, ["x axis stable", "y axis stable",
                     "island (both axes; followed, continuous)",
                     "Table-1 operating point"],
           loc="outside lower center", ncol=4, fontsize=9.5)
fig.suptitle("each zone is the crossing of an x band with a y band; "
             "the operating island is the overlap", fontsize=11)
plt.show()

## 5 · The four Table-1 zones in (a, q), gated against the paper

One $(a, q)$ panel per zone, drawn around each zone's own band. The table printed first is the gate: for each zone, the band the exact transfer matrix finds at $a = 0$ (center, width, and the band-limited resolving power $R_{BL} = q_c/\Delta q$, since $m \propto 1/q$ at fixed voltage) against the paper's $q$ and $R_{BL}$. A drawn chart that had not passed this gate would be a picture, not a result. One feature to expect below: at duty 0.5 (zone 3,3) the x and y axes see the same waveform, the wedge pinches onto the $a = 0$ line, and the panel's band is a single grid row.

In [ ]:
# the gate: band edges per zone from the same transfer matrix the
# panels use
print(f"{'zone':>5} {'duty':>7} {'q_tab':>7} {'q_found':>9} {'width':>9} "
      f"{'R_BL':>7} {'R_tab':>7}")
zone_bands = {}
for name, z in ZONES.items():
    lo, hi = QL.zone_band_a0(z["duty"], z["q"])
    qc, w = 0.5 * (lo + hi), hi - lo
    zone_bands[name] = (lo, hi)
    print(f"{name:>5} {z['duty']:7.4f} {z['q']:7.2f} {qc:9.4f} {w:9.5f} "
          f"{qc / w:7.0f} {z['r_bl']:7.0f}")
print("R_BL is the band-limited figure at a = 0: a transmission-defined "
      "resolution is lower (section 6).")

In [ ]:
# four (a, q) panels, each window sized by ITS band: +/- Q_WIN_WIDTHS
# band widths in q, +/- A_WIN_WIDTHS in a, so every zone resolves
probe_t0 = time.time()
QL.digital_stable(0.1, 8.0, 0.7)
per_point_s = time.time() - probe_t0
quote("zone panels, exact transfer matrices (measured per-point; the "
      "kernel is warm from section 3)", 4 * N_ZONE * N_ZONE,
      per_item_s=per_point_s)

fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.0))
for ax, (name, z) in zip(axes.ravel(), ZONES.items()):
    lo, hi = zone_bands[name]
    qc, w = 0.5 * (lo + hi), hi - lo
    qs = np.linspace(qc - Q_WIN_WIDTHS * w, qc + Q_WIN_WIDTHS * w, N_ZONE)
    aa = np.linspace(-A_WIN_WIDTHS * w, A_WIN_WIDTHS * w, N_ZONE)
    st = np.zeros((N_ZONE, N_ZONE), dtype=bool)
    for i, a in enumerate(aa):
        for j, q in enumerate(qs):
            st[i, j] = QL.digital_stable(a, q, z["duty"])
    # pcolormesh, not contourf: a band one grid row tall (zone 3,3 at
    # duty 0.5 pinches to the a = 0 line) vanishes from a contour but
    # keeps its cells in a mesh
    masked = np.ma.masked_where(~st, st.astype(float))
    ax.pcolormesh(qs, aa, masked, cmap="Blues", vmin=0, vmax=1.6,
                  shading="auto")
    ax.axhline(0, color="k", lw=0.5)
    # a dotted line, not a marker: an x at (q, 0) sits exactly on top of
    # a band that pinches to the a = 0 row (zone 3,3) and hides it
    ax.axvline(z["q"], color="C3", ls=":", lw=1.0)
    ax.set_title(f"zone {name}: duty {z['duty']:g}, Table-1 q = {z['q']:g}"
                 f"\n(window = band center +/- {Q_WIN_WIDTHS:g} widths)",
                 fontsize=8)
    ax.set_xlabel("q")
ax.set_ylabel("a")
fig.suptitle("Table-1 zones, x AND y stable (dotted line: the "
             "paper's q)", fontsize=10)
plt.tight_layout()
plt.show()

## 6 · The zone 3,1 band in flight

The drawn band is now flown: 2-D quad, digital drive at duty 0.835, the voltage stepped across the band in units of $q/q_c$, and at each step a ladder of `N_PHASE` waveform entry phases from a 10 µm start. The flown observable is **transmission** — the surviving fraction of the phase ladder — because a single ion's fate at these $q$ values depends on its entry phase: the formally stable orbit still swings through an intra-period amplitude gain of order $10^2$, so only part of the phase ladder fits inside the rods from any finite start. That phase selectivity is the mechanism behind the few-percent transmission the paper reports for the higher zones, and it is why $R_{BL}$ (the band-edge figure above) is an upper bound on what a transmission measurement shows.

In [ ]:
# the scan: transmission against q/qc; the first flight is timed and
# prices the rest (measured, not guessed)
lo, hi = zone_bands[ZONE]
qc = 0.5 * (lo + hi)
d_zone = ZONES[ZONE]["duty"]

def band_point(rel, pf):
    sp = QL.quad_spec(rel * qc * V_PER_Q, waveform="square", duty=d_zone,
                      phase_frac=pf, f_rf_mhz=F_RF_MHZ, mz=MZ,
                      x0=0.01, y0=0.007, t_max=T_MAX_US, dt_ns=DT_NS)
    _m, _tr, _ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
    return st["kind"] == 2

probe_t0 = time.time()
band_point(1.0, 0.0)
PER_FLIGHT_S = time.time() - probe_t0
quote(f"band-scan flights of {T_MAX_US} us at dt = {DT_NS} ns "
      f"(first flight measured)", len(REL_SCAN) * N_PHASE,
      per_item_s=PER_FLIGHT_S)

phases = np.linspace(0, 1, N_PHASE, endpoint=False)
trans = []
for rel in track(list(REL_SCAN), label="band scan"):
    trans.append(sum(band_point(rel, pf) for pf in phases) / N_PHASE)

print(f"zone {ZONE} flown at m/z {MZ:g}, {F_RF_MHZ} MHz, duty {d_zone:g}, "
      f"{qc*V_PER_Q:.1f} V center, 10 um start, {N_PHASE} phases/point:")
for rel, tr_ in zip(REL_SCAN, trans):
    inband = lo / qc <= rel <= hi / qc
    print(f"  q/qc {rel:.4f}: transmission {tr_:.2f}"
          + ("  (in the Meissner band)" if inband else ""))

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(REL_SCAN, trans, "o-", color="C0")
ax.axvspan(lo / qc, hi / qc, color="C2", alpha=0.25,
           label="Meissner band (x AND y stable)")
ax.set_xlabel("q / q_center")
ax.set_ylabel("transmission of the phase ladder")
ax.set_title(f"zone {ZONE} flown: m/z {MZ:g}, duty {d_zone:g}, "
             f"{V_PER_Q*qc:.0f} V, {F_RF_MHZ} MHz, 10 um start, "
             f"{N_PHASE} phases/point", fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# example trajectories: the same start at band center, one surviving and
# one lost entry phase, so the phase selectivity is visible as an ion
# path rather than asserted
fates = {}
for pf in phases:
    if len(fates) == 2:
        break
    sp = QL.quad_spec(qc * V_PER_Q, waveform="square", duty=d_zone,
                      phase_frac=pf, f_rf_mhz=F_RF_MHZ, mz=MZ,
                      x0=0.01, y0=0.007, t_max=T_MAX_US, dt_ns=DT_NS)
    _m, tr, ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
    key = "survives" if st["kind"] == 2 else "lost"
    if key not in fates:
        fates[key] = (pf, tr, ci, sp.geometry.width_mm / 2)
if len(fates) < 2:
    raise ValueError(f"band center produced only {sorted(fates)} across "
                     f"{N_PHASE} phases; both example fates are needed")
fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
for ax, (key, (pf, tr, ci, ctr)) in zip(axes, sorted(fates.items())):
    ax.plot(tr[:, ci["t"]], tr[:, ci["x"]] - ctr, lw=0.6, label="x")
    ax.plot(tr[:, ci["t"]], tr[:, ci["y"]] - ctr, lw=0.6, label="y")
    ax.set_ylabel("mm")
    ax.set_title(f"entry phase {pf:.3f} of the period: {key}", fontsize=9)
    ax.legend(fontsize=8)
axes[-1].set_xlabel("t (us)")
fig.suptitle(f"band-center ion paths, zone {ZONE} (10 um start)",
             fontsize=10)
plt.tight_layout()
plt.show()

### The entry-phase window

The example trajectories raise the right question: is the surviving entry phase a single sweet spot, or a window? The scan below flies a fine ladder of entry phases at the band center from the same 10 µm start. The answer is a **contiguous window**, not a point — its edges and width are printed, and the window is itself part of the transmission ceiling: of the ions that arrive on-axis, only the fraction born inside it survives. The window is phase of the drive waveform, so exploiting it is a timing problem, and the digital drive is the friendly case: the switch clock that makes the waveform can gate a pulsed injection to any chosen phase, a trick the digital ion trap community uses routinely. Under a sinusoidal drive the same gating needs a phase-locked pulser; with a digital drive it is the same clock.

In [ ]:
# transmission against entry phase at the band center, one flight per
# phase; the window is reported as phase fractions of the drive period
N_PHASE_FINE = 96
quote("entry-phase scan at band center (per-flight time from the band "
      "scan)", N_PHASE_FINE, per_item_s=PER_FLIGHT_S)
pf_grid = np.linspace(0.0, 1.0, N_PHASE_FINE, endpoint=False)
ok = np.array([band_point(1.0, pf)
               for pf in track(list(pf_grid), label="phase scan")])

# contiguous windows on the circle (the phase axis wraps)
edges = np.nonzero(np.diff(np.concatenate([ok, ok[:1]]).astype(int)))[0]
if ok.all():
    windows = [(0.0, 1.0)]
elif not ok.any():
    raise ValueError("no entry phase survives at band center -- the "
                     "band scan transmitted, so this scan disagrees "
                     "with it; investigate before trusting either")
else:
    starts = [(e + 1) % N_PHASE_FINE for e in edges if not ok[e]]
    stops = [(e + 1) % N_PHASE_FINE for e in edges if ok[e]]
    windows = []
    for st in starts:
        sp = min((x for x in stops if x > st),
                 default=stops[0] + N_PHASE_FINE)
        windows.append((pf_grid[st % N_PHASE_FINE],
                        pf_grid[sp % N_PHASE_FINE]
                        + (1.0 if sp >= N_PHASE_FINE else 0.0)))
print(f"surviving fraction of the phase ladder: {ok.mean():.2f}")
for lo_w, hi_w in windows:
    print(f"acceptance window: entry phase {lo_w:.3f} to {hi_w:.3f} of "
          f"the period (width {hi_w - lo_w:.3f}, edges located to "
          f"{1 / N_PHASE_FINE:.3f})")

fig, ax = plt.subplots(figsize=(7.6, 3.2), layout="constrained")
ax.step(pf_grid, ok.astype(int), where="post", color=CLR["x_band"],
        lw=1.6)
for lo_w, hi_w in windows:
    ax.axvspan(lo_w, min(hi_w, 1.0), color=CLR["flown_ok"], alpha=0.25)
    if hi_w > 1.0:
        ax.axvspan(0.0, hi_w - 1.0, color=CLR["flown_ok"], alpha=0.25)
ax.set_xlabel("entry phase (fraction of the drive period)")
ax.set_ylabel("survives 50 us")
ax.set_yticks([0, 1])
ax.set_title(f"zone {ZONE} band center: entry-phase acceptance from a "
             f"10 um start ({N_PHASE_FINE} flights)", fontsize=10)
plt.show()

## 7 · What the band demands of the clock

The integrator meets the square wave the same way hardware does: in discrete time. That puts **two distinct errors** on the duty cycle, and the band treats them very differently.

*Static edge offset.* When the step is **commensurate** with the drive — $dt = T/N$ with every switch edge landing exactly on a step — the waveform the ion feels has the exact duty every period, and the only cost of a coarser step is integration truncation over the Meissner equation's exponential-growth segments. The aligned ladder below keeps the band alive at every step down to $T/800$ — transmission varies non-monotonically with step size as truncation error interacts with the growth segments — and only the finest step, $T/14400$, reproduces the converged transmission.

*Per-period jitter.* When the step is **incommensurate**, each period's edges land up to one step away from their true positions, a duty error of order $dt/T$ that changes period to period. The jitter ladder below kills the band outright at every tested step — including $dt/T = 7\times10^{-5}$, four times smaller than the zone's **static** duty tolerance. A jittering duty is not a shifted band; it is parametric noise on a $10^{-3}$-wide resonance.

Both numbers transfer to hardware: a switch with a stable systematic skew moves the band (trim the frequency); a switch with cycle-to-cycle edge jitter at the $10^{-4}$ level has no band to find. This is also the deck-cutting rule (`make_digital_quad_deck`): the shipped dt is $T/N$ with the duty's edges on steps, sized inside the static tolerance — the first cut used an unaligned step and its flown band did not exist.

In [ ]:
# the two ladders at band center: edge-aligned steps (static offset
# only) against incommensurate steps (per-period duty jitter)
from quad_lib import zone_duty_tolerance
tol = zone_duty_tolerance(d_zone, ZONES[ZONE]["q"])
per_ns = 1e3 / F_RF_MHZ

def center_transmission(dt):
    n_ok = 0
    for pf in phases:
        sp = QL.quad_spec(qc * V_PER_Q, waveform="square", duty=d_zone,
                          phase_frac=pf, f_rf_mhz=F_RF_MHZ, mz=MZ,
                          x0=0.01, y0=0.007, t_max=T_MAX_US, dt_ns=dt)
        _m, _tr, _ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
        n_ok += st["kind"] == 2
    return n_ok / N_PHASE

n_jobs = (len(DT_ALIGNED_N) + len(DT_JITTER_NS)) * N_PHASE
quote("dt-ladder flights at band center (measured per-flight)",
      n_jobs, per_item_s=PER_FLIGHT_S)
aligned, jitter = [], []
for n_steps in track(DT_ALIGNED_N, label="aligned ladder"):
    dt = per_ns / n_steps
    aligned.append(center_transmission(dt))
    print(f"aligned   dt = T/{n_steps:<6d} = {dt:6.4f} ns: "
          f"transmission {aligned[-1]:.2f}")
for dt in track(DT_JITTER_NS, label="jitter ladder"):
    jitter.append(center_transmission(dt))
    print(f"jittering dt = {dt:6.4f} ns (dt/T = {dt/per_ns:.1e}): "
          f"transmission {jitter[-1]:.2f}")

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.semilogx([per_ns / n / per_ns for n in DT_ALIGNED_N], aligned, "o-",
            label="edge-aligned dt = T/N (static offset only)")
ax.semilogx([dt / per_ns for dt in DT_JITTER_NS], jitter, "s--", color="C3",
            label="incommensurate dt (per-period duty jitter)")
ax.axvline(tol, color="0.4", ls=":",
           label=f"static duty tolerance {tol:.1e} (transfer matrix)")
ax.set_xlabel("dt / T")
ax.set_ylabel("band-center transmission")
ax.set_title(f"zone {ZONE}: two clock error channels, flown", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8 · The 3-D deck of record, flown

The shipped deck's frequency is calibrated against its own solved field: a DC probe of the rod bases measures the axis curvature $A$ (Hessian eigenvalues of a 2-D quadratic fit, so the diagonal rod orientation cannot fool it), and $\Omega = \sqrt{2 e A / (m\, q_{zone})}$. That calibration moved the frequency 4.7% off the ideal-quad figure at the design $r_0$ — more than twenty band widths, which is why the deck reads the field and not the drawing.

The scan below flies a deterministic entry-phase ladder at frequencies around the calibrated value and records lifetimes. What it shows, at this deck's 0.5 mm voxelization and a 10 µm start: a clear lifetime **resonance** within 0.2% of the calibrated frequency — the band is where the calibration says — but no ion survives to transit, and the resonance frequency moves with the starting offset. That pattern is amplitude-dependent detuning: the voxelized field's effective curvature varies over the orbit's intra-period excursion by more than the band's $2\times10^{-3}$ width, so no single drive frequency keeps the whole orbit in-band. The integrator's edge handling is ruled out: the same scan with edge-aligned steps ($dt = T/20000$ at each frequency) reproduces the identical resonance with no survivors. The 2-D control at 0.1 mm pitch (section 6) transmits; the same control recalibrated at 0.5 mm pitch still transmits, so the gate is the 3-D voxelized field's linearity over the excursion, not lattice pitch alone. Mapping the acceptance of this deck against lattice `h` and rod profile is a study of its own and is left chartered, not run, in the read-out.

In [ ]:
# lifetime vs drive frequency in the 3-D deck: a deterministic phase
# ladder (both groups shifted together), point start near the axis,
# no axial energy, lifetimes as the observable
import contextlib
import io

def deck_lifetimes(rel_f, n_phase=N_PHASE_3D):
    lives = []
    for dph in np.linspace(0, 360, n_phase, endpoint=False):
        sp = SimSpec.from_json(str(DECK))
        for g in sp.geometry.rf_groups:
            g.frequency_hz = g.frequency_hz * rel_f
            g.phase_deg = (g.phase_deg + dph) % 360.0
        sp.source.distribution = "point"
        sp.source.n_ions = 1
        sp.source.x0_mm = sp.source.x0_mm + 0.01   # 10 um off the axis
        sp.source.y0_mm = sp.source.y0_mm + 0.007
        sp.source.z0_mm = 58.0                     # mid-rod
        sp.source.box_mm = [0.0, 0.0, 0.0]
        sp.source.ke_lo = 0.0
        sp.source.ke_hi = 0.0
        sp.source.seed = 3
        sp.integration.t_max_us = T_MAX_3D_US
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):   # solve-cache banner only
            _model, fly, _cols, _b = build_run(sp, record_budget_gb=RECORD_BUDGET_GB)
            _tr, st = fly(0)
        lives.append(st["tof"])
    return lives

probe_t0 = time.time()
deck_lifetimes(1.0, n_phase=1)
per_flight_3d_s = time.time() - probe_t0
quote("3-D deck lifetime scan (measured per-flight; the solve is cached)",
      len(REL_F_3D) * N_PHASE_3D, per_item_s=per_flight_3d_s)
rows3d = []
for rel in track(list(REL_F_3D), label="3-D freq scan"):
    lives = deck_lifetimes(rel)
    rows3d.append((rel, max(lives), np.median(lives),
                   sum(1 for t in lives if t > 0.8 * T_MAX_3D_US)))
for rel, mx, med, n_s in rows3d:
    print(f"f/f_deck {rel:.5f}: max life {mx:5.1f} us, median {med:4.1f}, "
          f"{n_s}/{N_PHASE_3D} near-full")

fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot([r[0] for r in rows3d], [r[1] for r in rows3d], "o-",
        label="max lifetime over the phase ladder")
ax.plot([r[0] for r in rows3d], [r[2] for r in rows3d], "s--", ms=4,
        label="median lifetime")
ax.axvline(1.0, color="C3", ls=":", label="calibrated deck frequency")
ax.set_xlabel("f / f_deck")
ax.set_ylabel("lifetime (us)")
ax.set_title(f"3-D deck, zone {ZONE}: lifetime resonance at 10 um start "
             f"(t_max {T_MAX_3D_US:g} us)", fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Summary

* The exact Meissner transfer matrix reproduces Table 1: the four zone centers land within 0.03% of the paper's $q$ values and the band-limited $R_{BL}$ figures within 5% for zones 3,1–3,3 (zone 4,1's published 12 500 against 11 905 found; its band is the narrowest and most sensitive to the duty's fourth decimal).
* The duty-$q$ map shows what the knob buys, and the zone anatomy shows its structure: each axis has its own band family moving oppositely with duty, two-axis stability exists only on the small $(d, q)$ islands where an $x$ band crosses a $y$ band (the zone label $m{,}n$ names the pair), and the measured islands span 0.001 to 0.015 in duty, with the operating $q$ sliding along the $x$ band inside each.
* Zone 3,1 flies in the 2-D quad: transmission is confined to the drawn band, with the phase ladder's partial survival (not band narrowness) setting the ceiling.
* The flown clock requirement matches the transfer matrix's static duty tolerance, and prices both the integrator step and a hardware switch's edge jitter.
* The 3-D deck's calibrated frequency puts a measured lifetime resonance where the band should be, but at 0.5 mm voxelization the field's effective curvature varies over the orbit excursion by more than the band width, and no sampled ion transits.

## Read-out

Every number and curve in this notebook is computed by ion_gym at run time — transfer matrices, calibrations, flights — with the published Table 1 serving only as the gate. The digital stability diagrams are reproduced from the exact two-segment transfer matrix and gated against that table; the zone 3,1 band, its phase-selective transmission, and its timing budget are verified in flight in the 2-D quad at the stated operating point (m/z 100, 1 MHz, duty 0.835, 10 µm start). The operational lesson the two flown sections share: a $10^{-3}$-wide band spends its error budget three ways at once — duty precision, clock resolution, and field linearity over the real orbit — and the third is the binding one for the voxelized 3-D deck. The chartered follow-up for that deck (not run here): an acceptance map against lattice `h` with the frequency recalibrated per `h`, and against rod profile, to establish what field quality higher-zone digital operation actually needs.